# Proyecto final · 05 · Observabilidad y optimización

**Componente 5 · Secciones 6 y 7 del examen.**

Lo que se evalúa:
- Una consulta a las tablas del sistema sobre **tu** proyecto: auditoría, linaje o corridas del job.
- Liquid clustering en tu tabla más grande, con `OPTIMIZE` y la evidencia en `DESCRIBE DETAIL` o `DESCRIBE HISTORY`.

Las tablas del sistema llegan con atraso: la auditoría, minutos; el linaje, unos 20. Si una consulta sale vacía, vuelve a correrla más tarde.

In [0]:
-- Consultar el linaje del proyecto para identificar las tablas
-- de origen y destino utilizadas durante las transformaciones.
SELECT DISTINCT
    source_table_full_name AS tabla_origen,
    target_table_full_name AS tabla_destino,
    entity_type AS tipo_ejecucion
FROM system.access.table_lineage
WHERE target_table_full_name LIKE 'pf_fallas.%'
  AND event_date >= current_date() - INTERVAL 7 DAYS
ORDER BY tabla_destino, tabla_origen;

In [0]:
-- Consultar el tamaño físico y los archivos de la tabla Bronze
-- para compararla con las otras tablas del pipeline.
DESCRIBE DETAIL pf_fallas.bronze.vehiculos_raw;

In [0]:
-- Consultar el tamaño físico de Silver para determinar
-- qué tabla conviene seleccionar para liquid clustering.
DESCRIBE DETAIL pf_fallas.silver.vehiculos;

In [0]:
-- Consultar el tamaño físico del resumen Gold
-- y completar la comparación entre las tres capas.
DESCRIBE DETAIL pf_fallas.gold.vehiculos_por_fabricante;

In [0]:
-- Consultar el tamaño de Silver para completar la comparación
-- y elegir la tabla más grande antes de aplicar liquid clustering.
DESCRIBE DETAIL pf_fallas.silver.vehiculos;

In [0]:
-- Configurar liquid clustering por fabricante en Bronze,
-- la tabla con mayor tamaño físico del pipeline.
ALTER TABLE pf_fallas.bronze.vehiculos_raw
CLUSTER BY (make);

In [0]:
-- Ejecutar OPTIMIZE para reorganizar los archivos de Bronze
-- según la configuración de liquid clustering.
OPTIMIZE pf_fallas.bronze.vehiculos_raw;

In [0]:
-- Verificar que clusteringColumns contiene make
-- y revisar el tamaño y la cantidad de archivos resultantes.
DESCRIBE DETAIL pf_fallas.bronze.vehiculos_raw;

In [0]:
-- Consultar las operaciones recientes de la tabla
-- para conservar evidencia de la ejecución de OPTIMIZE.
DESCRIBE HISTORY pf_fallas.bronze.vehiculos_raw LIMIT 10;

In [0]:
-- Consultar los modelos de un fabricante concreto.
-- El filtro por make justifica la columna elegida para clustering.
SELECT
    make,
    model,
    COUNT(*) AS registros_bronze
FROM pf_fallas.bronze.vehiculos_raw
WHERE make = 'TESLA'
GROUP BY make, model
ORDER BY registros_bronze DESC;

### TU RESPUESTA
¿Por qué elegiste esa columna para el clustering? ¿Qué consulta de tu proyecto se beneficia?

Se eligió `make` para liquid clustering porque permite filtrar los registros
por fabricante, como en la consulta de los modelos de TESLA.

Se aplicó sobre `bronze.vehiculos_raw`, la tabla de mayor tamaño físico
del pipeline antes de la optimización.

`DESCRIBE DETAIL` confirmó `make` como columna de clustering.
`OPTIMIZE` compactó dos archivos en uno y el tamaño pasó de `528.580`
a `510.208` bytes. El historial registra las operaciones realizadas.

La consulta a `system.access.table_lineage` mostró las relaciones entre
Bronze, Silver, la referencia de tipos y Gold, tanto en notebooks como en Jobs.

Debido al tamaño pequeño del dataset, estos resultados demuestran la
configuración y la compactación, pero no una mejora medible de velocidad.

In [0]:
-- Confirmar que la optimización conservó todos los registros
-- y los mismos vehículos únicos de Bronze.
SELECT
    COUNT(*) AS filas_totales,
    COUNT(DISTINCT vehicle_id) AS vehiculos_unicos
FROM pf_fallas.bronze.vehiculos_raw;